## 第3步 RLHF

运行环境：CPU 60G内存 1x16GB GPU P100

## 定义通用的参数

In [1]:
import argparse

args = argparse.ArgumentParser()

args.local_rank = 0
args.data_path = ["/mnt/Dahoas/rm-static"] 
args.data_split = "2,4,4" 
args.data_output_path = '/tmp/data_files'
args.max_seq_len = 512
args.num_train_epochs = 1 
args.seed = 1234 
args.per_device_train_batch_size = 1
args.per_device_eval_batch_size = 1
args.print_loss = True

args.model_name_or_path = '/mnt/chinese-alpaca-2-1.3b'
args.actor_model_name_or_path = "/mnt/chinese-alpaca-2-1.3b" 
args.critic_model_name_or_path = "/mnt/chinese-alpaca-2-1.3b" 

args.max_answer_seq_len = 256
args.max_prompt_seq_len = 256

compute_fp32_loss = True

args.per_device_generation_batch_size = 1

## 组织训练数据

In [2]:
from dschat.utils.data.data_utils import create_prompt_dataset
from transformers import LlamaTokenizer

tokenizer = LlamaTokenizer.from_pretrained(args.model_name_or_path)

train_phase = 3
prompt_train_dataset, _ = create_prompt_dataset(
    args.local_rank, args.data_path, args.data_split,
    args.data_output_path, train_phase, args.seed, tokenizer,
    args.max_seq_len, reload = True)

[2023-12-28 15:55:52,135] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)
Creating prompt dataset ['/mnt/Dahoas/rm-static'], reload=True
Creating dataset Dahoas_rm_static for train_phase=3 size=29601 filtered=901
Creating dataset Dahoas_rm_static for train_phase=3 size=1979 filtered=62


In [4]:
import torch

torch.set_printoptions(linewidth=100, edgeitems=5)

### prompt_train_dataset是训练样本数据，从数据集读出后进行反转

In [5]:
#tokenizer.pad_token_id = tokenizer.eos_token_id  #用llama2是要加上,chinese_llama2不需要

In [3]:
for prompt_input_ids, prompt_attention_mask, pad_token_id in prompt_train_dataset:
    print(f'input_ids：{prompt_input_ids}')
    print(f'attention_mask：{prompt_attention_mask}')
    print(f'{pad_token_id=}')
     
    break

input_ids：tensor([29901, 22137,  7900,    13,    13, 29973,   902,  1048,  1073,   881,
          306,  1683,  3099, 29901,  7889, 29950,    13,    13, 29889,  3186,
          278,  2820,  2305,  1784,   310, 12080,   278, 23051,  1183,   541,
        29892,   569,  3165, 14981, 29811,   471,   322, 29892,  8735,  1017,
        25273, 18677,   297, 10600,  2296, 29871, 29889, 12356,  2731,   322,
         6460,   278, 19912,  2834,  3353,   902, 10398,  2296, 29871, 29889,
        20604, 24819, 27813,   278, 15074,   471,  1584,   322, 29892,   664,
          902,   363,  3076,   324,  1035,   322, 24441,  1784,  4520,  8625,
         5061, 21869, 29871, 29889,  2834,   902,   310,  1791,   278,   363,
          963,   411,  3796,   322, 29892,   537,  2896,   310,  4314, 18971,
          278,   310,  8170,   278, 11091,  2296, 29871, 29889, 29947, 29946,
        29929, 29896, 29871,   297, 29892,  7513, 29892,   941,  7582,  3037,
          297,  6460,   278,   411,   664,   902,  468

In [4]:
from dschat.utils.data.data_utils import DataCollatorRLHF
from torch.utils.data import DataLoader

data_collator = DataCollatorRLHF(args.max_prompt_seq_len, 1)
prompt_train_dataloader = DataLoader(prompt_train_dataset, collate_fn=data_collator, 
                              batch_size=args.per_device_generation_batch_size)

## prompt_train_dataloader按最大序列长度进行左填空，将样本正过来

In [8]:
for step, batch_prompt in enumerate(prompt_train_dataloader):
    print(batch_prompt['prompt'])
    print(batch_prompt['prompt_att_mask'])
    break

tensor([[32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,     1, 29871,    13,
            13, 29950,  7889, 29901,  1724,   508,   366,  2649,   592,  1048,   278,  2834,   310,
         21869,  5061,  8625,    13,    13,  7900, 22137, 29901, 21869,  5061,  8625,   471,  6345,
           297, 21805,   423,   297, 29871, 29896, 29929, 29896, 29900, 29892,   322,  1183,  4689,


In [9]:
print(tokenizer.decode(batch_prompt['prompt'][0]))

<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><s>

Human: What can you tell me about the life of Mother Teresa

Assistant: Mother Teresa was born in Albania in 1910, and she began her work with the poor in Calcutta, India, in 1948.  She founded the Order of the Missionaries of Charity, and worked with them for the rest of her life.  Mother Teresa received many awards and accolades for her work, and even was awarded the Nobel Peace Prize.  She spent her whole life helping the poor and destitute.  She lived in extreme poverty herself, and was incredibly humble, but she touched the lives of many people

## 同时生成actor、ref、ema、critic、reward model
演员模型actor_model、参考模型ref_model采用LlamaForCausalLM，得到预测分数

评论家模型critic_model、奖励模型reward_model采用LlamaModel，得到嵌入向量

In [5]:
from dschat.utils.model.reward_model import RewardModel
from transformers import LlamaForCausalLM, LlamaModel

actor_model = LlamaForCausalLM.from_pretrained(args.actor_model_name_or_path, device_map="cpu")
ref_model = LlamaForCausalLM.from_pretrained(args.actor_model_name_or_path, device_map="cpu")
#actor_model_ema = LlamaForCausalLM.from_pretrained(args.actor_model_name_or_path, device_map="auto")

critic_model = LlamaModel.from_pretrained(args.critic_model_name_or_path, device_map="cpu")
critic_model = RewardModel(
        critic_model,
        tokenizer,
        num_padding_at_beginning = 0,
        compute_fp32_loss = compute_fp32_loss)

reward_model = LlamaModel.from_pretrained(args.critic_model_name_or_path, device_map="cpu")
reward_model = RewardModel(
        reward_model,
        tokenizer,
        num_padding_at_beginning = 0,
        compute_fp32_loss = compute_fp32_loss)

/home/llama/jupyter/jup_notebook/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:362: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/llama/jupyter/jup_notebook/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:367: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(


## 演员模型推理

In [6]:
import torch

def _generate_sequence(prompts, mask, step):
    max_min_length = args.max_answer_seq_len + prompts.shape[1]

    with torch.no_grad():
        seq = actor_model.generate(
            prompts,
            attention_mask=mask,
            max_length=max_min_length,
            pad_token_id=tokenizer.pad_token_id)

    batch_size = seq.shape[0]
    prompt_length = prompts.shape[1]
    ans = seq[:, prompt_length:]
    valid_ans_len = (ans != tokenizer.pad_token_id).sum(dim=-1)

    out_seq = []
    for i in range(batch_size):
        out_seq.append(seq[i:i + 1])
    out_seq = torch.cat(out_seq, dim=0)  # concat output in the batch dim

    return out_seq, prompt_length
    
for step, batch_prompt in enumerate(prompt_train_dataloader):
    prompts = batch_prompt['prompt']
    mask = batch_prompt['prompt_att_mask']

    seq, prompt_length = _generate_sequence(prompts, mask, step)

    print(seq)
    break

/home/llama/jupyter/jup_notebook/lib/python3.10/site-packages/transformers/generation/utils.py:1411: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(


tensor([[32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,     1, 29871,
            13,    13, 29950,  7889, 29901,  1724,   508,   366,  2649,   592,
          1048,   278,  2834,   310, 21869,  5061,  8625,    13,    13,  7900,
         22137, 29901, 21869,  5061,  8625,   471,  6345,   297, 21805,   423,
           297, 29871, 29896, 29929, 29896, 29900, 2

In [10]:
print(tokenizer.decode(seq[0]))

<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><s>

Human: What can you tell me about the life of Mother Teresa

Assistant: Mother Teresa was born in Albania in 1910, and she began her work with the poor in Calcutta, India, in 1948.  She founded the Order of the Missionaries of Charity, and worked with them for the rest of her life.  Mother Teresa received many awards and accolades for her work, and even was awarded the Nobel Peace Prize.  She spent her whole life helping the poor and destitute.  She lived in extreme poverty herself, and was incredibly humble, but she touched the lives of many people

## 对演员模型推理结果打分
#### actor_model => logprobs
#### ref_model => ref_logprobs

In [7]:
import torch.nn.functional as F
import torch

def gather_log_probs(logits, labels):
    log_probs = F.log_softmax(logits, dim=-1)
    log_probs_labels = log_probs.gather(dim=-1, index=labels.unsqueeze(-1))
    return log_probs_labels.squeeze(-1)
    
pad_token_id = tokenizer.pad_token_id
attention_mask = seq.not_equal(pad_token_id).long()

output = actor_model(seq, attention_mask=attention_mask)
output_ref = ref_model(seq, attention_mask=attention_mask)

logits = output.logits
logits_ref = output_ref.logits

logprobs = gather_log_probs(logits[:, :-1, :], seq[:, 1:])
ref_logprobs = gather_log_probs(logits_ref[:, :-1, :], seq[:,1:])

In [8]:
import torch

torch.set_printoptions(linewidth=100, edgeitems=5, precision = 2, profile="default")

In [20]:
logits

tensor([[[-2.01, -2.86,  2.57, -1.46, -1.72,  ..., -2.33, -2.20, -1.66, -2.20, -1.97],
         [-2.00, -2.86,  2.56, -1.47, -1.72,  ..., -2.33, -2.19, -1.65, -2.20, -1.96],
         [-2.00, -2.85,  2.55, -1.47, -1.72,  ..., -2.32, -2.18, -1.65, -2.20, -1.95],
         [-1.99, -2.85,  2.55, -1.47, -1.71,  ..., -2.31, -2.17, -1.65, -2.19, -1.95],
         [-1.99, -2.85,  2.54, -1.47, -1.71,  ..., -2.31, -2.16, -1.64, -2.19, -1.94],
         ...,
         [-3.23, -3.44,  5.65, -2.17, -1.02,  ...,  0.83,  1.61, -4.08, -1.34, -0.98],
         [-3.05, -4.89,  3.87, -1.83, -2.57,  ...,  0.39, -0.47, -2.56, -4.88, -1.60],
         [-3.23, -5.15,  2.95, -2.27, -1.73,  ..., -1.87, -0.67, -3.33, -6.98, -1.97],
         [-2.13, -3.25,  6.74,  0.03, -0.29,  ...,  0.70,  1.55, -0.27, -3.08,  2.73],
         [-3.40, -2.62,  3.82, -1.72, -0.23,  ..., -4.02,  0.69,  0.03, -2.67, -3.33]]],
       grad_fn=<UnsafeViewBackward0>)

In [9]:
print(f'{logits.size()=}\n{logprobs.size()=}')

logits.size()=torch.Size([1, 512, 55296])
logprobs.size()=torch.Size([1, 511])


In [13]:
print(f'{logprobs[0,511-1]=}\n{ref_logprobs[0,511-1]=}')

logprobs[0,511-1]=tensor(-0.0011, grad_fn=<SelectBackward0>)
ref_logprobs[0,511-1]=tensor(-0.0011, grad_fn=<SelectBackward0>)


In [22]:
with torch.no_grad():
    lables = seq[:, 1:]
    t = lables[0,510]
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    print(log_probs[0,510,t])

tensor(-0.0011)


## 计算奖励分数
### reware_model => reward_score

In [9]:
pad_token_id = tokenizer.pad_token_id
attention_mask = seq.not_equal(pad_token_id).long()

_reward_score = reward_model.forward_value(
    seq, attention_mask,
    prompt_length=prompt_length)

reward_score = _reward_score['chosen_end_scores']
print(f'{reward_score = }')
values = _reward_score['values']
print(f'{values.size()=}')

reward_score = tensor([1.59], grad_fn=<StackBackward0>)
values.size()=torch.Size([1, 512])


#### reward_model.forward_value函数代码
#### 第3步 在开始填充 [0 0 0 0 prompt, answer, 0 0 0 0 ] 
序列seq包括提示和回答两部分，seq_len是整个序列的长度，prompt_length是提示的长度

In [10]:
input_ids = seq
PAD_ID = tokenizer.pad_token_id

bs = values.size(0)
seq_len = input_ids.shape[1]
chosen_end_scores = [] 
for i in range(bs):
    input_id = input_ids[i]
    value = values[i]

    c_inds = (input_id[prompt_length:] == PAD_ID).nonzero()
    
    # 这里仅使用序列的回答部分，不关注开始的填空
    c_ind = c_inds[0].item() + prompt_length if len(
        c_inds) > 0 else seq_len
    chosen_end_scores.append(value[c_ind - 1])

### critic_model => values

In [11]:
values = critic_model.forward_value(
    seq, attention_mask, return_value_only=True).detach()[:, :-1]

### actor_loss

In [12]:
cliprange = 0.2
cliprange_value = 0.2

start = prompts.size()[-1] - 1
action_mask = attention_mask[:, 1:]
advantages = 1

def actor_loss_fn(logprobs, old_logprobs, advantages, mask):
    ## policy gradient loss
    log_ratio = (logprobs - old_logprobs) * mask
    ratio = torch.exp(log_ratio)
    pg_loss1 = -advantages * ratio
    pg_loss2 = -advantages * torch.clamp(ratio, 1.0 - cliprange,
                                         1.0 + cliprange)
    pg_loss = torch.sum(torch.max(pg_loss1, pg_loss2) * mask) / mask.sum()
    return pg_loss

batch = {'input_ids': seq, "attention_mask": attention_mask}
actor_prob = actor_model(**batch, use_cache=False).logits
actor_log_prob = gather_log_probs(actor_prob[:, :-1, :], seq[:, 1:])
actor_loss = actor_loss_fn(actor_log_prob[:, start:],
                                logprobs[:, start:], advantages,
                                action_mask[:, start:])

### critic_loss

In [ ]:
old_values = values
returns

def critic_loss_fn(values, old_values, returns, mask):
    ## value loss
    values_clipped = torch.clamp(
        values,
        old_values - cliprange_value,
        old_values + cliprange_value,
    )
    if compute_fp32_loss:
        values = values.float()
        values_clipped = values_clipped.float()
    vf_loss1 = (values - returns)**2
    vf_loss2 = (values_clipped - returns)**2
    vf_loss = 0.5 * torch.sum(
        torch.max(vf_loss1, vf_loss2) * mask) / mask.sum()
    return vf_loss
    
value = critic_model.forward_value(**batch,
                                        return_value_only=True,
                                        use_cache=False)[:, :-1]
critic_loss = critic_loss_fn(value[:, start:], old_values[:,start:],                                                                       
                                  returns, action_mask[:, start:])

## => advantages, returns

#### 计算奖励的函数

In [20]:
kl_ctl = 0.1
clip_reward_value = 5

def compute_rewards(prompts, log_probs, ref_log_probs, reward_score,
                    action_mask):

    kl_divergence_estimate = -kl_ctl * (log_probs - ref_log_probs)
    rewards = kl_divergence_estimate
    start = prompts.shape[1] - 1
    ends = start + action_mask[:, start:].sum(1) + 1
    
    reward_clip = torch.clamp(reward_score, -clip_reward_value,
                              clip_reward_value)
    batch_size = log_probs.shape[0]

    for j in range(batch_size):
        rewards[j, start:ends[j]][-1] += reward_clip[j]

    return rewards

#### 获取优势和回报的函数

In [26]:
gamma = 1.0
lam = 0.95

def get_advantages_and_returns(values, rewards, start):
    # Adopted from https://github.com/CarperAI/trlx/blob/main/trlx/models/modeling_ppo.py#L134
    lastgaelam = 0
    advantages_reversed = []
    length = rewards.size()[-1]
    for t in reversed(range(start, length)):
        nextvalues = values[:, t + 1] if t < length - 1 else 0.0
        delta = rewards[:, t] + gamma * nextvalues - values[:, t]
        lastgaelam = delta + gamma * lam * lastgaelam
        advantages_reversed.append(lastgaelam)
        print(f'{t=}\t{lastgaelam=}\t{nextvalues=}\t{delta=}\t{advantages_reversed}')
        
    advantages = torch.stack(advantages_reversed[::-1], dim=1)
    returns = advantages + values[:, start:]

    print(f'{advantages=}\t{returns=}')
    return advantages.detach(), returns

In [27]:
start = prompts.size()[-1] - 1
action_mask = attention_mask[:, 1:]

old_values = values
action_mask = attention_mask
with torch.no_grad():
    old_rewards = compute_rewards(prompts, logprobs,
                                       ref_logprobs, reward_score,
                                       action_mask)
    
    ends = start + action_mask[:, start:].sum(1) + 1
    # 将reward和value清零，否则会出错
    for i in range(old_rewards.shape[0]):
        old_rewards[i, ends[i]:] = 0
        old_values[i, ends[i]:] = 0
    advantages, returns = get_advantages_and_returns(
        old_values, old_rewards, start)

t=510	lastgaelam=tensor([1.44])	nextvalues=0.0	delta=tensor([1.44])	[tensor([1.44])]
t=509	lastgaelam=tensor([1.73])	nextvalues=tensor([0.16])	delta=tensor([0.36])	[tensor([1.44]), tensor([1.73])]
t=508	lastgaelam=tensor([1.66])	nextvalues=tensor([-0.21])	delta=tensor([0.02])	[tensor([1.44]), tensor([1.73]), tensor([1.66])]
t=507	lastgaelam=tensor([2.65])	nextvalues=tensor([-0.23])	delta=tensor([1.08])	[tensor([1.44]), tensor([1.73]), tensor([1.66]), tensor([2.65])]
t=506	lastgaelam=tensor([2.49])	nextvalues=tensor([-1.30])	delta=tensor([-0.03])	[tensor([1.44]), tensor([1.73]), tensor([1.66]), tensor([2.65]), tensor([2.49])]
t=505	lastgaelam=tensor([1.23])	nextvalues=tensor([-1.27])	delta=tensor([-1.14])	[tensor([1.44]), tensor([1.73]), tensor([1.66]), tensor([2.65]), tensor([2.49]), tensor([1.23])]
t=504	lastgaelam=tensor([1.17])	nextvalues=tensor([-0.13])	delta=tensor([0.01])	[tensor([1.44]), tensor([1.73]), tensor([1.66]), tensor([2.65]), tensor([2.49]), tensor([1.23]), tensor([1.17